# 02 — Beat BLSTM (EXP-018), Colab / PyTorch

Learned beat tracking: a bidirectional LSTM predicts a per-frame **beat
activation** from log-mel frames; our **own** tempo+DP phase decoder turns the
activation into beat times. No `librosa.beat.beat_track`, no madmom — the
musical decision (tempo + phase) stays our code.

**Run in Google Colab** (GPU runtime recommended). Steps:
1. Run Setup (installs deps, clones the repo).
2. Point `DATA_ROOT` at a folder containing `train/` and
   `train_extra_tempobeats/` (each with `.wav` + `.beats.gt`). In Colab, put the
   data on Google Drive and mount it.
3. Run through training; the eval cell reports beat F1 **before**
   (autocorrelation baseline) vs **after** (BLSTM). Success: F1 > 0.45 on the
   held-out 20%.
4. The save cell writes `models/beat_blstm.pt`. Copy it back into the repo to
   wire inference (see EXP-018 in the experiment log; `src/detectors.py` change
   is done separately after review).


In [ ]:
# === Setup (Colab-ready) ===
import sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "librosa", "soundfile", "mir_eval"], check=False)
    REPO = Path("amp-challenge")
    if not REPO.exists():
        subprocess.run(["git", "clone",
                        "https://github.com/8asic/amp2026-onset-beat-tempo.git",
                        str(REPO)], check=True)
else:
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO))

import numpy as np
import librosa
import mir_eval
import torch
import torch.nn as nn

from src.config import config
from src.detectors import BeatTracker
from src.utils import load_beats_gt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "| repo:", REPO)

In [ ]:
# === Data: extract zips from Drive to fast local disk, then locate dirs ===
import zipfile

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/amp_data")   # <-- folder where you uploaded the zips
    WORK = Path("/content/data"); WORK.mkdir(exist_ok=True)
    for name in ["train", "train_extra_tempobeats"]:
        z, dest = ZIP_DIR / f"{name}.zip", WORK / name
        if dest.exists():
            print("already extracted:", dest); continue
        assert z.exists(), f"Missing {z} - upload {name}.zip to {ZIP_DIR}"
        print("extracting", z, "...")
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
    base = WORK
else:
    base = REPO / "data" / "processed"

def find_dir_with(suffix, root):
    root = Path(root)
    if not root.exists():
        return None
    for p in [root] + [d for d in root.rglob("*") if d.is_dir()]:
        if any(p.glob(f"*{suffix}")):
            return p
    return None

train_dir = find_dir_with(".beats.gt", base / "train")
extra_dir = find_dir_with(".beats.gt", base / "train_extra_tempobeats")
print("train_dir:", train_dir)
print("extra_dir:", extra_dir)
assert train_dir and extra_dir, "Could not locate .beats.gt dirs - check ZIP_DIR / uploads"

In [ ]:
# === Collect beat-annotated files (127 main + 696 extra = 823) ===
def collect(d, corpus):
    d = Path(d); items = []
    if not d.exists():
        return items
    for wav in sorted(d.glob("*.wav")):
        gtp = d / f"{wav.stem}.beats.gt"
        if gtp.exists():
            beats = load_beats_gt(gtp)
            if beats is not None and len(beats):
                items.append((str(wav), np.asarray(beats, dtype=float), corpus))
    return items

main_files = collect(train_dir, "c127")          # the 127 main corpus (test-like)
extra_files = collect(extra_dir, "extra")        # the 696 supplementary corpus
files = main_files + extra_files
print(f"main(c127): {len(main_files)}  extra: {len(extra_files)}  total: {len(files)}")
assert files, "No files found - check DATA_ROOT path."

In [ ]:
# === Features (log-mel, 81 bands) + per-frame beat labels (cached) ===
SR, HOP, NFFT, NMELS = 22050, 512, 2048, 81
FPS = SR / HOP
FMIN, FMAX = config.audio.onset_fmin, config.audio.onset_fmax

def logmel(y):
    m = librosa.feature.melspectrogram(y=y, sr=SR, n_fft=NFFT, hop_length=HOP,
                                       n_mels=NMELS, fmin=FMIN, fmax=FMAX)
    return np.log1p(m).T.astype(np.float32)            # (T, NMELS)

def beat_labels(beats, T):
    lab = np.zeros(T, dtype=np.float32)
    for f in np.round(beats * FPS).astype(int):
        for dd in (-1, 0, 1):
            if 0 <= f + dd < T:
                lab[f + dd] = 1.0
    return lab

CACHE = REPO / "experiments" / ".cache_beat"
CACHE.mkdir(parents=True, exist_ok=True)

data = []
for i, (wav, beats, corpus) in enumerate(files):
    stem = Path(wav).stem
    cp = CACHE / f"{stem}.npz"
    if cp.exists():
        d = np.load(cp)
        X, y = d["X"], d["y"]
    else:
        y_audio, _ = librosa.load(wav, sr=SR)
        X = logmel(y_audio)
        y = beat_labels(beats, X.shape[0])
        np.savez(cp, X=X, y=y)
    data.append({"stem": stem, "X": X, "y": y, "beats": beats, "wav": wav, "corpus": corpus})
    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(files)} features")
print("features ready:", len(data))

In [ ]:
# === Split + standardization (stats from TRAIN only) ===
# EXP-018 FAIR TEST: train on the 696 extra files only, evaluate on all 127 c127
# files (the test-like corpus the model NEVER trained on). This is the honest
# generalization check. Set False for the shippable model trained on all 823.
TRAIN_ON_EXTRA_ONLY = True

if TRAIN_ON_EXTRA_ONLY:
    train_set = [d for d in data if d["corpus"] == "extra"]
    val_set   = [d for d in data if d["corpus"] == "c127"]
else:
    rng = np.random.default_rng(0)
    order = rng.permutation(len(data))
    n_val = int(0.2 * len(data))
    val_ids = set(order[:n_val].tolist())
    train_set = [data[i] for i in range(len(data)) if i not in val_ids]
    val_set = [data[i] for i in range(len(data)) if i in val_ids]

print(f"train {len(train_set)}  val {len(val_set)}  (extra_only={TRAIN_ON_EXTRA_ONLY})")
assert train_set and val_set, "empty split"

allX = np.concatenate([d["X"] for d in train_set])
MU = allX.mean(0); SD = allX.std(0) + 1e-8
def norm(X):
    return (X - MU) / SD

In [ ]:
# === Dataset / loaders (batch_size=1: variable-length sequences) ===
class BeatDS(torch.utils.data.Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        d = self.items[i]
        return (torch.from_numpy(norm(d["X"])),     # (T, NMELS)
                torch.from_numpy(d["y"]))           # (T,)

train_dl = torch.utils.data.DataLoader(BeatDS(train_set), batch_size=1, shuffle=True)
val_dl = torch.utils.data.DataLoader(BeatDS(val_set), batch_size=1, shuffle=False)

In [ ]:
# === Model: bidirectional LSTM -> per-frame beat-activation logit ===
class BeatBLSTM(nn.Module):
    def __init__(self, n_mels=NMELS, hidden=128, layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(n_mels, hidden, layers, batch_first=True,
                            bidirectional=True,
                            dropout=dropout if layers > 1 else 0.0)
        self.fc = nn.Linear(2 * hidden, 1)
    def forward(self, x):                 # x: (B, T, n_mels)
        h, _ = self.lstm(x)
        return self.fc(h).squeeze(-1)     # (B, T) logits

HIDDEN, LAYERS = 128, 2
model = BeatBLSTM(hidden=HIDDEN, layers=LAYERS).to(DEVICE)
print(sum(p.numel() for p in model.parameters()), "params")

In [ ]:
# === Train (BCE + grad-clip + ReduceLROnPlateau + early stopping) ===
POS_WEIGHT = torch.tensor([6.0], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=3)
EPOCHS = 60
ES_PATIENCE = 8

best_val, best_state, since_best = float("inf"), None, 0
for ep in range(1, EPOCHS + 1):
    model.train(); tr = 0.0
    for X, y in train_dl:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=3.0)
        opt.step()
        tr += loss.item()
    model.eval(); va = 0.0
    with torch.no_grad():
        for X, y in val_dl:
            X, y = X.to(DEVICE), y.to(DEVICE)
            va += crit(model(X), y).item()
    va /= len(val_dl)
    sched.step(va)
    if va < best_val:
        best_val, since_best = va, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        since_best += 1
    print(f"epoch {ep:2d}  train {tr/len(train_dl):.4f}  val {va:.4f}  "
          f"lr {opt.param_groups[0]['lr']:.2e}")
    if since_best >= ES_PATIENCE:
        print(f"early stop at epoch {ep} (no val improvement for {ES_PATIENCE} epochs)")
        break

if best_state is not None:
    model.load_state_dict(best_state)   # restore best-val weights
print(f"best val loss {best_val:.4f}")

In [ ]:
# === Decode beats from the activation using OUR tempo+DP (rules-clean) ===
bt = BeatTracker()

def decode(activation):
    env = np.asarray(activation, dtype=np.float64)
    N = len(env)
    tempo = bt._estimate_tempo_from_env(env, FPS)          # our autocorrelation tempo
    lag_min = max(1, int(np.ceil(60.0 / config.beat.tempo_max * FPS)))
    lag_max = int(np.floor(60.0 / config.beat.tempo_min * FPS))
    lag = int(np.clip(round(60.0 * FPS / tempo), lag_min, lag_max))
    frames = bt._dp_beats(env, lag, N)                     # our DP phase search
    return np.array(frames, dtype=int) / FPS

@torch.no_grad()
def activation(X):
    model.eval()
    t = torch.from_numpy(norm(X)).unsqueeze(0).to(DEVICE)
    return torch.sigmoid(model(t)).squeeze(0).cpu().numpy()

# AFTER: BLSTM activation -> our tempo+DP
blstm_f = [mir_eval.beat.f_measure(d["beats"], decode(activation(d["X"]))) for d in val_set]

# BEFORE: autocorrelation baseline (current log-mel-flux ODF -> same tempo+DP)
base_f = []
for d in val_set:
    y_audio, _ = librosa.load(d["wav"], sr=SR)
    est, _ = bt.track(y_audio, SR)
    base_f.append(mir_eval.beat.f_measure(d["beats"], est))

print(f"Autocorrelation baseline beat F1 (val): {np.mean(base_f):.4f}")
print(f"BLSTM beat F1 (val):                    {np.mean(blstm_f):.4f}")
print(f"Delta: {np.mean(blstm_f) - np.mean(base_f):+.4f}")
print(f"Success (>0.45): {'YES' if np.mean(blstm_f) > 0.45 else 'no'}")

In [ ]:
# === Save weights + everything inference needs ===
# Fair-test model (extra-only) saves to beat_blstm_extra.pt so it does NOT
# overwrite the shippable all-823 model (beat_blstm.pt).
fname = "beat_blstm_extra.pt" if TRAIN_ON_EXTRA_ONLY else "beat_blstm.pt"
out = REPO / "models" / fname
out.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "mu": MU, "sd": SD,
    "n_mels": NMELS, "hidden": HIDDEN, "layers": LAYERS,
    "sr": SR, "hop": HOP, "n_fft": NFFT, "fmin": FMIN, "fmax": FMAX,
}, out)
print("saved", out)
# In Colab, download it back to your machine / repo:
# from google.colab import files; files.download(str(out))